# 노트북 4. 컨텍스트 윈도우와 토큰

> Phase 2 — 제어: 모델 동작을 원하는 대로 다루기

토큰은 LLM의 화폐입니다.
비용도 토큰, 성능 한계도 토큰, 응답 품질도 토큰에 달렸습니다.

**학습 목표**
- 토큰의 실체(서브워드 단위)를 이해하고 직접 측정할 수 있다
- 한국어와 영어의 토큰 효율 차이를 설명할 수 있다
- 컨텍스트 윈도우의 개념과 한계를 이해한다
- 멀티턴 대화에서의 토큰 누적과 비용 구조를 계산할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 토큰 실체 + count_tokens + 컨텍스트 윈도우 + 비용 | 읽고 실행 |
| Part 2 — 실습 | 토큰 측정 + 비용 계산 + 윈도우 초과 재현 | 직접 작성 |

In [1]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "gemini-2.5-flash"
print("환경 설정 완료")

환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 토큰의 실체

LLM은 텍스트를 단어 단위가 아니라 **토큰(token)** 단위로 처리합니다.
토큰은 **서브워드(subword)** 단위입니다.
하나의 단어가 여러 토큰으로 쪼개질 수도 있고, 짧은 단어는 하나의 토큰이 될 수도 있습니다.

예시:
- `"hello"` → 1토큰
- `"unhappiness"` → `["un", "happiness"]` → 2토큰
- `"인공지능"` → 여러 토큰 (한국어는 더 많은 토큰 소비)

> 핵심: 토큰은 LLM이 텍스트를 이해하는 최소 단위입니다.
> 비용, 속도, 컨텍스트 한계 모두 토큰 수로 결정됩니다.

### count_tokens() API

Gemini는 `client.models.count_tokens()` API로 텍스트의 토큰 수를 직접 측정할 수 있습니다.
실제 API 호출을 하지 않으므로 비용이 들지 않습니다.

아래 코드는 간단한 문장의 토큰 수를 측정합니다.

In [2]:
# count_tokens()로 토큰 수 측정
result = client.models.count_tokens(
    model=MODEL,
    contents="Hello, world!",
)

print(f"'Hello, world!' → {result.total_tokens} 토큰")

'Hello, world!' → 5 토큰


여러 문장의 토큰 수를 비교해봅시다.
단어 수와 토큰 수는 반드시 일치하지 않습니다.

In [3]:
# 다양한 영어 텍스트의 토큰 수
english_texts = [
    "Hello",
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Artificial intelligence is transforming the world.",
    "supercalifragilisticexpialidocious",  # 매우 긴 단어
]

print(f"{'텍스트':<55} {'단어수':>5} {'토큰수':>5}")
print("-" * 70)
for text in english_texts:
    token_count = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    word_count = len(text.split())
    print(f"{text:<55} {word_count:>5} {token_count:>5}")

텍스트                                                       단어수   토큰수
----------------------------------------------------------------------
Hello                                                       1     2
Hello, world!                                               2     5
The quick brown fox jumps over the lazy dog.                9    11
Artificial intelligence is transforming the world.          6     8
supercalifragilisticexpialidocious                          1    11


### 한국어 vs 영어 토큰 효율

한국어는 영어보다 **토큰 효율이 낮습니다**.
같은 의미를 전달하는 데 더 많은 토큰을 소비합니다.
이것은 한국어 사용자에게 비용과 컨텍스트 윈도우 측면에서 불리합니다.

아래 코드는 같은 의미의 한국어/영어 문장의 토큰 수를 비교합니다.

In [4]:
# 동일 의미 한국어/영어 토큰 비교
pairs = [
    ("Hello", "안녕하세요"),
    ("What is artificial intelligence?", "인공지능이란 무엇인가요?"),
    ("The capital of South Korea is Seoul.", "대한민국의 수도는 서울입니다."),
    ("Python is a popular programming language.", "파이썬은 인기 있는 프로그래밍 언어입니다."),
    ("Machine learning models learn patterns from data.", "머신러닝 모델은 데이터에서 패턴을 학습합니다."),
]

print(f"{'영어':<50} {'EN':>4} {'한국어':<35} {'KO':>4} {'배율':>5}")
print("-" * 100)
for en, ko in pairs:
    en_tokens = client.models.count_tokens(model=MODEL, contents=en).total_tokens
    ko_tokens = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
    ratio = ko_tokens / en_tokens if en_tokens > 0 else 0
    print(f"{en:<50} {en_tokens:>4} {ko:<35} {ko_tokens:>4} {ratio:>5.1f}x")

영어                                                   EN 한국어                                   KO    배율
----------------------------------------------------------------------------------------------------
Hello                                                 2 안녕하세요                                  2   1.0x
What is artificial intelligence?                      6 인공지능이란 무엇인가요?                         10   1.7x
The capital of South Korea is Seoul.                  9 대한민국의 수도는 서울입니다.                       9   1.0x
Python is a popular programming language.             8 파이썬은 인기 있는 프로그래밍 언어입니다.               14   1.8x
Machine learning models learn patterns from data.     9 머신러닝 모델은 데이터에서 패턴을 학습합니다.             14   1.6x


> 핵심: 한국어는 영어 대비 약 1.5~2배의 토큰을 사용합니다.
> 같은 대화를 한국어로 하면 비용이 더 많이 들고, 컨텍스트 윈도우도 더 빨리 차게 됩니다.

### 토큰 수 어림 규칙

API 호출 없이 토큰 수를 대략적으로 추정하는 경험 규칙이 있습니다.
정확하지는 않지만, 비용이나 컨텍스트 사용량을 빠르게 어림할 때 유용합니다.

| 언어 | 어림 규칙 | 근거 |
|------|----------|------|
| 영어 | 1토큰 ≈ 4글자 (또는 0.75단어) | 라틴 알파벳 기반 서브워드 |
| 한국어 | 1토큰 ≈ 1~2글자 | 유니코드 다바이트 문자, 조합형 |
| 코드 | 단어 수와 비슷 | 식별자, 기호가 개별 토큰 |

아래 코드는 이 어림 규칙이 실제 토큰 수와 얼마나 차이 나는지 확인합니다.

In [5]:
# 어림 규칙 vs 실제 토큰 수 비교
test_cases = [
    ("영어", "The quick brown fox jumps over the lazy dog.", lambda t: len(t) / 4),
    ("영어", "Artificial intelligence is the future of technology.", lambda t: len(t) / 4),
    ("한국어", "인공지능은 기술의 미래입니다.", lambda t: len(t) / 1.5),
    ("한국어", "오늘 날씨가 매우 좋습니다.", lambda t: len(t) / 1.5),
    ("코드", "def fibonacci(n):\n    if n <= 1: return n", lambda t: len(t.split())),
]

print(f"{'유형':<6} {'텍스트':<50} {'어림':>5} {'실제':>5} {'오차':>6}")
print("-" * 80)
for lang, text, estimator in test_cases:
    estimated = estimator(text)
    actual = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    error = ((estimated - actual) / actual) * 100
    print(f"{lang:<6} {text:<50} {estimated:>5.0f} {actual:>5} {error:>+5.0f}%")

유형     텍스트                                                   어림    실제     오차
--------------------------------------------------------------------------------
영어     The quick brown fox jumps over the lazy dog.          11    11    +0%
영어     Artificial intelligence is the future of technology.    13     9   +44%
한국어    인공지능은 기술의 미래입니다.                                      11    11    -3%
한국어    오늘 날씨가 매우 좋습니다.                                       10     8   +25%
코드     def fibonacci(n):
    if n <= 1: return n              8    17   -53%


### 다양한 유형의 텍스트 토큰화

텍스트 유형에 따라 토큰 효율이 달라집니다.
코드, 숫자, 특수 문자는 일반 텍스트와 다른 토큰화 패턴을 보입니다.

아래 코드는 다양한 유형의 텍스트가 어떻게 토큰화되는지 비교합니다.

In [6]:
# 다양한 유형의 텍스트 토큰화
varied_texts = {
    "일반 영어": "The weather is nice today.",
    "일반 한국어": "오늘 날씨가 좋습니다.",
    "파이썬 코드": "def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "JSON 데이터": '{"name": "Kim", "age": 30, "city": "Seoul"}',
    "숫자 나열": "1234567890 9876543210 1111111111",
    "URL": "https://www.example.com/api/v2/users?page=1&limit=10",
    "긴 반복": "ha " * 50,  # "ha ha ha ..." 50번
}

print(f"{'유형':<12} {'글자수':>6} {'토큰수':>6} {'글자/토큰':>8}")
print("-" * 40)
for label, text in varied_texts.items():
    tokens = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    chars = len(text)
    ratio = chars / tokens if tokens > 0 else 0
    print(f"{label:<12} {chars:>6} {tokens:>6} {ratio:>8.1f}")

유형              글자수    토큰수    글자/토큰
----------------------------------------
일반 영어            26      7      3.7
일반 한국어           12      7      1.7
파이썬 코드           84     35      2.4
JSON 데이터         43     20      2.1
숫자 나열            32     33      1.0
URL              52     24      2.2
긴 반복            150     52      2.9


## 1.2 count_tokens() 심화

`count_tokens()`는 단순 텍스트뿐 아니라 실제 API 호출과 동일한 형태의 입력을 측정할 수 있습니다.
`system_instruction`을 포함한 전체 토큰 수도 계산 가능합니다.

### system_instruction 포함 토큰 수

실제 API 호출에서는 `system_instruction`도 토큰으로 소비됩니다.
system prompt가 길면 매 호출마다 그만큼의 토큰이 추가됩니다.

아래 코드는 system_instruction의 토큰 수를 확인합니다.

여기선 system_prompt 단독의 기여도를 보기 위해서 실제 generate_content를 통해서 계산하도록 합니다.

In [7]:
# system_instruction 포함 토큰 수 측정
system_prompt = "당신은 친절한 한국어 도우미입니다. 간결하게 답변하세요."
user_message = "배움이란 무엇인가요 간결하게 답변하세요?"

# system_instruction 없이
tokens_without = client.models.generate_content(
    model=MODEL,
    contents=user_message,
)

# system_instruction 포함
tokens_with = client.models.generate_content(
    model=MODEL,
    contents=user_message,
    config={"system_instruction": system_prompt}
)


print(f"user만: {tokens_without.usage_metadata.total_token_count} 토큰")
print(f"system + user: {tokens_with.usage_metadata.total_token_count} 토큰")

user만: 1397 토큰
system + user: 183 토큰


### 멀티턴 대화의 토큰 측정

멀티턴 대화에서는 매 호출마다 전체 이력이 전송됩니다.
`count_tokens()`로 실제 전송될 토큰 수를 사전에 확인할 수 있습니다.

아래 코드는 대화가 쌓이면서 토큰 수가 어떻게 증가하는지 보여줍니다.

In [8]:
from google.genai.types import Content, Part

# 가상 멀티턴 대화 이력
conversation = [
    Content(role="user", parts=[Part(text="안녕하세요, 저는 김철수입니다.")]),
    Content(role="model", parts=[Part(text="안녕하세요, 김철수님! 무엇을 도와드릴까요?")]),
    Content(role="user", parts=[Part(text="파이썬 공부를 시작하려고 합니다.")]),
    Content(role="model", parts=[Part(text="좋은 선택입니다! 파이썬은 초보자에게 추천하는 언어입니다.")]),
    Content(role="user", parts=[Part(text="어디서부터 시작하면 좋을까요?")]),
]

# 턴별 누적 토큰 수 확인
print(f"{'턴':>3} {'누적 메시지':>10} {'토큰 수':>8}")
print("-" * 25)
for i in range(1, len(conversation) + 1):
    partial = conversation[:i]
    tokens = client.models.count_tokens(model=MODEL, contents=partial).total_tokens
    print(f"{i:>3} {len(partial):>10}개 {tokens:>8}")

  턴     누적 메시지     토큰 수
-------------------------
  1          1개        9
  2          2개       23
  3          3개       33
  4          4개       53
  5          5개       63


### Content/Part 구조체와 토큰

지금까지 `contents`에 문자열을 직접 전달했지만, 실제로는 `Content` 객체와 `Part` 객체로 구조화할 수 있습니다.
문자열 전달과 구조체 전달의 토큰 수가 동일한지 확인해봅시다.

아래 코드는 같은 텍스트를 다른 방식으로 전달했을 때 토큰 수를 비교합니다.

In [9]:
# 같은 텍스트를 다른 형태로 전달했을 때 토큰 수 비교
text = "오늘 날씨가 어떤가요?"

# 방법 1: 문자열 직접 전달
t1 = client.models.count_tokens(model=MODEL, contents=text).total_tokens

# 방법 2: Content 객체로 전달
t2 = client.models.count_tokens(
    model=MODEL,
    contents=Content(role="user", parts=[Part(text=text)]),
).total_tokens

# 방법 3: 리스트로 감싸서 전달
t3 = client.models.count_tokens(
    model=MODEL,
    contents=[Content(role="user", parts=[Part(text=text)])],
).total_tokens

print(f"문자열 직접: {t1} 토큰")
print(f"Content 객체: {t2} 토큰")
print(f"리스트 감싸기: {t3} 토큰")
print(f"\n→ 전달 방식에 관계없이 동일한 텍스트는 동일한 토큰 수입니다.")

문자열 직접: 9 토큰
Content 객체: 9 토큰
리스트 감싸기: 9 토큰

→ 전달 방식에 관계없이 동일한 텍스트는 동일한 토큰 수입니다.


## 1.3 컨텍스트 윈도우

**컨텍스트 윈도우(Context Window)**는 모델이 한 번에 처리할 수 있는 최대 토큰 수입니다.
입력 토큰과 출력 토큰의 합이 이 한계를 넘을 수 없습니다.

```
입력 토큰 + 출력 토큰 <= 컨텍스트 윈도우
```

### Gemini 모델별 컨텍스트 윈도우

https://ai.google.dev/gemini-api/docs/long-context?hl=ko

| 모델 | 컨텍스트 윈도우 | 최대 출력 토큰 |
|------|----------------|---------------|
| gemini-2.5-flash | 1,048,576 (약 100만) | 65,536 |
| gemini-2.5-pro | 1,048,576 (약 100만) | 65,536 |
| gemini-2.0-flash | 1,048,576 (약 100만) | 8,192 |

100만 토큰은 약 70만 단어, 책 약 10권 분량에 해당합니다.

> 핵심: 컨텍스트 윈도우가 크다고 해서 항상 좋은 것은 아닙니다.
> 긴 컨텍스트에서는 모델의 주의력이 분산되는 **Lost in the Middle** 현상이 발생할 수 있습니다.
> 중간에 위치한 정보는 앞이나 뒤에 있는 정보보다 덜 잘 인식됩니다.

### 컨텍스트 사용률 계산

실제 호출에서 컨텍스트 윈도우를 얼마나 사용했는지 확인해봅시다.

아래 코드는 호출 후 사용된 토큰과 윈도우 한계를 비교합니다.

In [10]:
# 컨텍스트 사용률 확인
CONTEXT_WINDOW = 1_048_576  # gemini-2.5-flash

response = client.models.generate_content(
    model=MODEL,
    contents="대한민국의 역사를 간략히 설명해주세요.",
)

usage = response.usage_metadata
total_used = usage.prompt_token_count + usage.candidates_token_count
usage_pct = (total_used / CONTEXT_WINDOW) * 100

print(f"입력 토큰: {usage.prompt_token_count:,}")
print(f"출력 토큰: {usage.candidates_token_count:,}")
print(f"합계: {total_used:,}")
print(f"컨텍스트 윈도우: {CONTEXT_WINDOW:,}")
print(f"사용률: {usage_pct:.4f}%")

입력 토큰: 12
출력 토큰: 1,204
합계: 1,216
컨텍스트 윈도우: 1,048,576
사용률: 0.1160%


### usage_metadata 상세 분석

API 응답의 `usage_metadata`에는 여러 필드가 포함되어 있습니다.
어떤 정보를 제공하는지 전체를 확인해봅시다.

아래 코드는 `usage_metadata`의 모든 필드를 출력합니다.

In [11]:
# usage_metadata 전체 필드 확인
response = client.models.generate_content(
    model=MODEL,
    contents="파이썬의 장점을 3가지 알려주세요.",
)

usage = response.usage_metadata
print("=== usage_metadata 필드 ===")
print(f"prompt_token_count:     {usage.prompt_token_count:>8,}  # 입력 토큰 수")
print(f"candidates_token_count: {usage.candidates_token_count:>8,}  # 출력 토큰 수")
print(f"total_token_count:      {usage.total_token_count:>8,}  # 입력 + 출력")

# thinking_token_count는 모델이 thinking을 지원할 때만 존재
thinking_tokens = getattr(usage, 'thoughts_token_count', None)
if thinking_tokens is not None:
    print(f"thoughts_token_count:   {thinking_tokens:>8,}  # 추론(thinking) 토큰")
else:
    print(f"thoughts_token_count:   해당 없음 (thinking 비활성화 상태)")

print(f"\n검증: prompt({usage.prompt_token_count}) + candidates({usage.candidates_token_count}) = {usage.prompt_token_count + usage.candidates_token_count}")
print(f"       total_token_count = {usage.total_token_count}")

=== usage_metadata 필드 ===
prompt_token_count:           14  # 입력 토큰 수
candidates_token_count:      326  # 출력 토큰 수
total_token_count:         1,363  # 입력 + 출력
thoughts_token_count:      1,023  # 추론(thinking) 토큰

검증: prompt(14) + candidates(326) = 340
       total_token_count = 1363


단일 호출에서는 윈도우의 극히 일부만 사용합니다.
하지만 멀티턴 대화가 누적되면 상황이 달라집니다.

### 컨텍스트 윈도우 초과 시

입력이 컨텍스트 윈도우를 초과하면 API가 에러를 반환합니다.
이 에러를 직접 확인해보는 것이 중요합니다.

아래 코드는 매우 긴 텍스트를 전송하여 윈도우 한계를 테스트합니다.
(Gemini 2.5 Flash의 윈도우가 100만 토큰으로 매우 크기 때문에,
먼저 긴 텍스트의 토큰 수를 측정하고, 윈도우 대비 사용률을 확인합니다.)

In [12]:
# 매우 긴 텍스트 생성 후 토큰 수 확인
long_text = "이것은 매우 긴 텍스트입니다. " * 5000  # 약 5000번 반복

token_count = client.models.count_tokens(
    model=MODEL,
    contents=long_text,
).total_tokens

print(f"반복 텍스트 길이: {len(long_text):,} 글자")
print(f"토큰 수: {token_count:,}")
print(f"윈도우 대비: {(token_count / CONTEXT_WINDOW) * 100:.2f}%")

반복 텍스트 길이: 85,000 글자
토큰 수: 45,003
윈도우 대비: 4.29%


### Thinking 토큰과 비용

Gemini 2.5 모델은 **Thinking(추론)** 기능을 지원합니다.
Thinking이 활성화되면 모델이 답변 전에 내부적으로 "생각"하는 과정을 거치며, 이 과정에서도 토큰이 소비됩니다.

Thinking 관련 토큰은 3종류입니다:

| 토큰 종류 | 설명 | 과금 |
|----------|------|------|
| Input tokens | 사용자 입력 + system prompt | 입력 단가 |
| Output tokens | 실제 응답 텍스트 | 출력 단가 |
| Thinking tokens | 내부 추론 과정 | 출력 단가와 동일 |

> 주의: thinking_budget를 높게 설정하면 추론 토큰이 크게 증가할 수 있습니다.
> 단순한 질문에는 thinking을 비활성화하는 것이 비용 효율적입니다.
> Thinking에 대해서는 노트북 10에서 자세히 다룹니다.

아래 코드는 Thinking 활성화/비활성화 시 토큰 사용량 차이를 비교합니다.

In [13]:
# Thinking 활성화/비활성화 토큰 비교
question = "7 + 13은 얼마인가요?"

# thinking 비활성화 (budget=0)
resp_no_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 0}},
)

# thinking 활성화 (기본값)
resp_think = client.models.generate_content(
    model=MODEL,
    contents=question,
    config={"thinking_config": {"thinking_budget": 1024}},
)

u1 = resp_no_think.usage_metadata
u2 = resp_think.usage_metadata

# thinking 토큰 확인
think_tokens = getattr(u2, 'thoughts_token_count', 0) or 0

print(f"{'항목':<20} {'Thinking OFF':>15} {'Thinking ON':>15}")
print("-" * 55)
print(f"{'입력 토큰':<20} {u1.prompt_token_count:>15,} {u2.prompt_token_count:>15,}")
print(f"{'출력 토큰':<20} {u1.candidates_token_count:>15,} {u2.candidates_token_count:>15,}")
print(f"{'Thinking 토큰':<20} {'0':>15} {think_tokens:>15,}")
print(f"{'전체 토큰':<20} {u1.total_token_count:>15,} {u2.total_token_count:>15,}")

항목                      Thinking OFF     Thinking ON
-------------------------------------------------------
입력 토큰                             11              11
출력 토큰                             12              12
Thinking 토큰                        0              64
전체 토큰                             23              87


---

### 생각해보기

1. 한국어가 영어보다 토큰 효율이 낮다면, 한국어 서비스를 운영할 때 비용을 줄이는 방법은 무엇이 있을까요?
2. 컨텍스트 윈도우가 100만 토큰이라면 사실상 무제한이라고 볼 수 있을까요? 여전히 윈도우를 의식해야 하는 이유는 무엇일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 4-1: 다양한 텍스트의 토큰 수 측정

아래 텍스트들의 토큰 수를 `count_tokens()`로 측정하고 표로 출력하세요.

**요구사항**
1. `client.models.count_tokens()`를 사용
2. 아래 5가지 텍스트를 측정:
   - `"Hello, world!"`
   - `"안녕하세요, 세계!"`
   - `"def hello(): print('hi')"`
   - `"12345 67890 11111"`
   - `"https://www.google.com/search?q=gemini"`
3. 각 텍스트의 글자 수와 토큰 수를 출력

In [14]:
# TODO: 5가지 텍스트의 토큰 수를 측정하세요
texts = [
    "Hello, world!",
    "안녕하세요, 세계!",
    "def hello(): print('hi')",
    "12345 67890 11111",
    "https://www.google.com/search?q=gemini",
]

# ---------- 정답 ----------
print(f"{'텍스트':<45} {'글자수':>6} {'토큰수':>6}")
print("-" * 60)
for text in texts:
    tokens = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"{text:<45} {len(text):>6} {tokens:>6}")

print("TODO를 완성해주세요")

텍스트                                              글자수    토큰수
------------------------------------------------------------
Hello, world!                                     13      5
안녕하세요, 세계!                                        10      5
def hello(): print('hi')                          24      8
12345 67890 11111                                 17     18
https://www.google.com/search?q=gemini            38     15
TODO를 완성해주세요


### 실습 4-2: 한국어/영어 토큰 효율 비교

동일한 의미의 한국어/영어 문장 쌍을 3개 이상 작성하고 토큰 수를 비교하세요.

**요구사항**
1. 한국어/영어 쌍을 최소 3개 작성 (직접 작성)
2. 각 쌍의 토큰 수를 측정
3. 한국어가 영어 대비 몇 배의 토큰을 사용하는지 계산
4. 평균 배율을 출력

In [15]:
# TODO: 한국어/영어 쌍을 작성하고 토큰 효율을 비교하세요
pairs = [
    # ("English sentence", "한국어 문장"),
]  # 최소 3개의 쌍을 작성하세요

# ---------- 정답 ----------
pairs = [
    ("I like coffee.", "저는 커피를 좋아합니다."),
    ("The weather is cold today.", "오늘 날씨가 춥습니다."),
    ("Please recommend a good restaurant.", "좋은 식당을 추천해주세요."),
]

if pairs:
    ratios = []
    print(f"{'영어':<40} {'EN':>4} {'한국어':<25} {'KO':>4} {'배율':>6}")
    print("-" * 85)
    for en, ko in pairs:
        en_t = client.models.count_tokens(model=MODEL, contents=en).total_tokens
        ko_t = client.models.count_tokens(model=MODEL, contents=ko).total_tokens
        ratio = ko_t / en_t if en_t > 0 else 0
        ratios.append(ratio)
        print(f"{en:<40} {en_t:>4} {ko:<25} {ko_t:>4} {ratio:>6.2f}x")
    print(f"\n평균 배율: {sum(ratios)/len(ratios):.2f}x")
else:
    print("TODO를 완성해주세요")

영어                                         EN 한국어                         KO     배율
-------------------------------------------------------------------------------------
I like coffee.                              5 저는 커피를 좋아합니다.                7   1.40x
The weather is cold today.                  7 오늘 날씨가 춥습니다.                 9   1.29x
Please recommend a good restaurant.         7 좋은 식당을 추천해주세요.               9   1.29x

평균 배율: 1.32x


### 실습 4-3: system_instruction의 토큰 비용 확인

긴 system prompt와 짧은 system prompt가 토큰 수에 어떤 영향을 미치는지 확인하세요.

**요구사항**
1. 짧은 system prompt: "간결하게 답변하세요."
2. 긴 system prompt: 5줄 이상의 상세한 규칙 포함 (직접 작성)
3. 동일한 질문("파이썬이란?")에 대해 각각 `count_tokens()` 실행
4. 토큰 수 차이를 출력

In [16]:
question = "파이썬이란?"

# TODO 1: 짧은 system prompt
short_system = ""  # 이 문자열을 작성하세요

# TODO 2: 긴 system prompt (5줄 이상)
long_system = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
short_system = "간결하게 답변하세요."
long_system = (
    "당신은 10년 경력의 소프트웨어 엔지니어입니다.\n"
    "초보자가 이해할 수 있도록 쉽게 설명하세요.\n"
    "반드시 예시 코드를 포함하세요.\n"
    "각 개념을 설명할 때 비유를 사용하세요.\n"
    "답변은 500자 이내로 작성하세요.\n"
    "마지막에 핵심 요약을 한 줄로 추가하세요."
)

# 검증
if short_system and long_system:
    t_short = client.models.count_tokens(
        model=MODEL, contents=short_system + question,
    ).total_tokens

    t_long = client.models.count_tokens(
        model=MODEL, contents=long_system + question,
    ).total_tokens

    print(f"짧은 system: {t_short} 토큰")
    print(f"긴 system:   {t_long} 토큰")
    print(f"차이: {t_long - t_short} 토큰 (매 호출마다 추가됨)")
else:
    print("TODO를 완성해주세요")

짧은 system: 12 토큰
긴 system:   86 토큰
차이: 74 토큰 (매 호출마다 추가됨)
